# 02 — What the four reasoning methods learned

Notebook 01 read four columns of scores. This one opens the models behind them.

PI, RRF, GPTree and RRM each produce an artifact you can read: a list of
policies, a shortlist of questions, a tree, a set of mined rules. That is the
practical difference between reasoning ML and a gradient-boosted model over
embeddings. The learned object is text, so you can audit it, disagree with it,
and show it to someone who does not write code.

For each method: what it does, the artifact it produced on VCBench, and the
public-split score that artifact earned.

Nothing here needs the raw dataset or a network. The one exception is the last
code cell, a single call to a local `qwen2.5-coder:14b` so you can watch the
model that produced all of this answer a question. It skips itself if Ollama is
not running.

In [1]:
import json
import os
import socket
import sys
import textwrap
from collections import defaultdict
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score


def find_example_root(start: Path | None = None) -> Path:
    """Walk up from `start` (default: the CWD) to the example dir holding precomputed/."""
    start = (start or Path.cwd()).resolve()
    for parent in (start, *start.parents):
        for candidate in (parent, parent / "examples" / "vcbench-movie-local"):
            if (candidate / "precomputed").is_dir():
                return candidate
    raise FileNotFoundError(
        "Could not find the example directory (the one containing precomputed/) "
        f"above {start}. Run this notebook from inside the example."
    )


EXAMPLE_ROOT = find_example_root()
PRECOMPUTED = EXAMPLE_ROOT / "precomputed"
MODELS = EXAMPLE_ROOT / "models"


def score_ranking(labels, scores) -> dict[str, float]:
    return {
        "ROC-AUC": roc_auc_score(labels, scores),
        "PR-AUC": average_precision_score(labels, scores),
    }


public_metrics: dict[str, dict[str, float]] = {}


def report(name: str, method: str) -> None:
    """Score one method's shipped public-split predictions and remember the result."""
    df = pd.read_csv(PRECOMPUTED / f"vcbench_public_{method}_scores.csv")
    metrics = score_ranking(df["label"], df["score"])
    public_metrics[name] = metrics
    print(
        f"{name} · VCBench public (n={len(df):,}) · "
        f"ROC-AUC {metrics['ROC-AUC']:.4f} · PR-AUC {metrics['PR-AUC']:.4f}"
    )


print("example root:", EXAMPLE_ROOT.name)
print("bundles:     ", ", ".join(sorted(p.name for p in (MODELS / "vcbench").iterdir())))

example root: vcbench-movie-local
bundles:      gptree, pi, rrf, rrm


## PolicyInduction (PI)

PI asks the LLM to read labelled examples and write down rules of thumb, in
plain English, that separate the classes. It keeps a small set of them, ten
here, then puts every policy to the model as a yes/no question about every
founder. Each founder becomes a ten-bit vector, and a logistic regression fitted
out of fold turns the vector into a score.

Two things are worth noticing below. The policies are readable, which is the
whole point. And the fitted weights argue with them: a policy the LLM proposed
confidently can end up with a negative coefficient, which is the regression
saying the policy is either wrong on this data or already covered by another
one.

In [2]:
policies = pd.read_csv(MODELS / "vcbench" / "pi" / "pi_local_policies.csv")
meta = json.loads((MODELS / "vcbench" / "pi" / "pi_local_meta.json").read_text())
manifest = json.loads((MODELS / "vcbench" / "pi" / "policy_induction.json").read_text())

# `feature_order` lists the policy ids in the order the regression saw them.
weights = dict(zip(manifest["feature_order"], meta["coefficients"][0]))
policies["weight"] = policies["policy_id"].astype(str).map(weights).round(3)

print(f"induced from {meta['n_founders']:,} founders by "
      f"{manifest['gen_llmc'][0]['model']} at temperature {manifest['gen_temperature']}")
print(f"logistic combiner: intercept {meta['intercept'][0]:.3f}, "
      f"decision threshold {meta['threshold']:.2f}\n")

with pd.option_context("display.max_colwidth", None):
    display(policies[["policy_id", "weight", "policy"]].set_index("policy_id"))

report("PI", "pi")

induced from 4,500 founders by qwen2.5-coder:14b at temperature 0.0
logistic combiner: intercept -2.544, decision threshold 0.21



,weight,policy
policy_id,,
0,-0.087,"If a founder has an MSc or higher degree from a top-ranked institution (QS rank <= 10) and at least 4 years of professional experience in relevant fields, they are likely to succeed."
1,-0.051,"Founders with leadership roles such as CEO, CIO, or Director in their previous positions are more likely to succeed."
2,0.152,"A combination of technical expertise (e.g., software engineering, computer science) and business acumen (e.g., management, investment) increases the likelihood of success."
3,0.774,Education from institutions ranked within the top 100 globally is a strong indicator of potential success.
4,-0.307,Founders with experience in multiple industries or sectors are more adaptable and may have a higher chance of success.
5,0.282,A founder's professional experience should include roles that demonstrate both technical skills and business leadership.
6,0.308,"If a founder has a background in finance, technology, or software development, they are more likely to succeed."
7,0.187,"Education in fields such as engineering, physics, or applied sciences can be advantageous for certain startups."
8,0.908,Founders with a history of innovation and product development are more likely to achieve significant success.


PI · VCBench public (n=4,500) · ROC-AUC 0.6763 · PR-AUC 0.1831


## RRF

RRF works from a shortlist of yes/no questions rather than rules. The LLM
generates candidates over several seeds, a selection pass keeps the ones that
carry signal, and every surviving question is put to the model about every
founder. As with PI, the answer vector is collapsed by an out-of-fold logistic
regression.

The shortlist shipped here has 16 questions, all written by the same local qwen
model. `expected_direction` is the generator's guess at which answer points to
success, `source` records which generation seed the question came from. Neither
field constrains the combiner, which fits its own weights.

In [3]:
shortlist = pd.DataFrame(
    json.loads((MODELS / "vcbench" / "rrf" / "coder14b_shortlist.json").read_text())
)

print(f"{len(shortlist)} questions, from {shortlist['source'].nunique()} generation seeds\n")
with pd.option_context("display.max_colwidth", None):
    display(shortlist.set_index("qid"))

report("RRF", "rrf")

16 questions, from 4 generation seeds



,text,expected_direction,source
qid,,,
c01_prior_exit,"Does the founder have prior exits (IPOs, acquisitions) under their belt?",1,qwen2.5-coder-14b/seed_0
c02_big_exit,Has the founder overseen acquisitions valued at over $150M?,1,qwen2.5-coder-14b/seed_3
c03_qs_top50_uni,Did the founder attend a QS top 50 university?,1,qwen2.5-coder-14b/seed_0
c04_phd,Is the founder's highest degree a PhD or higher?,1,qwen2.5-coder-14b/seed_0
c05_prior_ceo_1000,"Has the founder held a CEO role in a company with over 1,000 employees?",1,qwen2.5-coder-14b/seed_0
c06_vc_pe,Has the founder been involved in venture capital or private equity?,1,qwen2.5-coder-14b/seed_0
c07_prior_leadership,Did the founder hold any leadership roles before starting their current startup?,1,qwen2.5-coder-14b/seed_0
c08_10k_employees,"Has the founder worked in companies with over 10,000 employees?",1,qwen2.5-coder-14b/seed_0
c09_advisor_board,Did the founder serve as an advisor or board member for startups?,1,qwen2.5-coder-14b/seed_0


RRF · VCBench public (n=4,500) · ROC-AUC 0.6604 · PR-AUC 0.1689


## GPTree

GPTree is a decision tree whose splits are questions instead of numeric
thresholds. At each node it asks the LLM for candidate questions with a fixed
set of answer choices, scores each candidate by how well it separates the
labels (gini, as in CART), keeps the best one, and recurses down every answer
branch. This tree was grown to `max_depth=3` with `min_samples_leaf=3`.

A leaf predicts the training success rate of the founders who landed in it, so
the printout below is the entire model, root question to leaf probability.

It also shows why GPTree is the weakest of the four on held-out data. Most
leaves sit near the 9% base rate, and the ones that do not are small: the 0.364
leaf holds 11 founders. A tree three questions deep over a 9%-positive dataset
runs out of signal before it runs out of depth.

In [4]:
tree = json.loads((MODELS / "vcbench" / "gptree" / "gptree.json").read_text())
nodes = {node["id"]: node for node in tree["nodes"]}
children = defaultdict(list)
for node in tree["nodes"]:
    if node["parent_id"] is not None:
        children[node["parent_id"]].append(node["id"])

POSITIVE = tree["classes"][1]


def render(node_id: int, depth: int = 0) -> None:
    node = nodes[node_id]
    pad = "   " * depth
    if node["parent_id"] is not None:
        print(f"{pad}├─ {node['label']}")
    question = node.get("question")
    if question:
        print(f"{pad}   ? {question['value']}")
        for child in children[node_id]:
            render(child, depth + 1)
    else:
        counts = node.get("class_distribution") or {}
        n = sum(counts.values())
        rate = counts.get(POSITIVE, 0) / n if n else float("nan")
        print(f"{pad}   → leaf · n={n:>5,} · P({POSITIVE}) = {rate:.3f}")


leaves = [node for node in tree["nodes"] if not children[node["id"]]]
print(f"{len(tree['nodes'])} nodes, {len(leaves)} leaves, "
      f"max_depth={tree['params']['max_depth']}, "
      f"base rate {nodes[0]['class_distribution'][POSITIVE] / sum(nodes[0]['class_distribution'].values()):.3f}\n")
render(0)
print()

report("GPTree", "gptree")

51 nodes, 35 leaves, max_depth=3, base rate 0.090

   ? What is the primary industry of the founder's startup?
   ├─ Industrial & Agricultural Machinery Manufacturing
      ? How long has the founder been leading the startup?
      ├─ >3 years
         → leaf · n=   32 · P(successful) = 0.000
      ├─ <=2 years
         ? What is the employee size range of the startup or company led by the founder?
         ├─ 11-50 employees
            → leaf · n=   32 · P(successful) = 0.000
         ├─ 1001+ employees
            → leaf · n=   83 · P(successful) = 0.108
         ├─ 201-500 employees
            → leaf · n=   67 · P(successful) = 0.090
      ├─ 2-3 years
         ? In what industry does the startup of this founder operate?
         ├─ Financial Services
            → leaf · n=   27 · P(successful) = 0.222
         ├─ Electronic & Precision Equipment Manufacturing
            → leaf · n=   60 · P(successful) = 0.133
         ├─ Other
            → leaf · n=   56 · P(successful) = 0.0

## RRM (Reasoned Rule Mining)

RRM lets the LLM reason freely over labelled examples, extracts IF/THEN rules
out of that reasoning, filters them by the perplexity of the model's own text
(a rough confidence proxy), then compiles the survivors into a single decision
policy. Prediction runs the compiled policy, not the raw rule list, with a Platt
calibrator on top.

The VCBench bundle ships with `rules: []`, on purpose. The mined rules tracked
the founder prose closely enough that some of them named real products, which
would undo the anonymisation the benchmark rests on. Dropping them costs
nothing at predict time, since the predict path only reads the compiled policy
and the calibrator, and it keeps the benchmark meaningful: a held-out split
stops measuring anything once the answers can be read off the shipped model.

Movie's rules ship in full and stand in below. They are aggregate patterns over
public plot summaries, 346 of them over 2,188 films, so no single film can be
recovered from one, and they are the most legible illustration of what RRM
produces.

In [5]:
rrm = json.loads((MODELS / "vcbench" / "rrm" / "reasoned_rule_mining.json").read_text())

print(f"VCBench RRM: {len(rrm['rules'])} rules shipped (emptied), "
      f"compiled policy {len(rrm['policy'])} chars, Platt calibrator "
      f"{(MODELS / 'vcbench' / 'rrm' / rrm['ens_platt_file']).exists()}")
print("\nThe compiled policy, which is what actually predicts:\n")
for line in rrm["policy"].splitlines():
    print(textwrap.fill(line, 88, subsequent_indent="    ") if line.strip() else "")

movie_rrm = json.loads((MODELS / "movie" / "rrm" / "reasoned_rule_mining.json").read_text())
rules = pd.DataFrame(movie_rrm["rules"])
print(f"\nMovie RRM: {len(rules)} mined rules · "
      f"{dict(rules['outcome'].value_counts())} · "
      f"perplexity {rules['perplexity'].min():.2f}–{rules['perplexity'].max():.2f}")
print("The six the model was most confident in (lowest perplexity):\n")
with pd.option_context("display.max_colwidth", None):
    display(rules.nsmallest(6, "perplexity").reset_index(drop=True))

report("RRM", "rrm")

VCBench RRM: 0 rules shipped (emptied), compiled policy 1028 chars, Platt calibrator True

The compiled policy, which is what actually predicts:

For YES: IF ((Diverse Educational Background AND Extensive Career Experience IN
    Executive Leadership Roles AND Strategic Versatility) OR (PhD and MS in Relevant
    Field AND Long Research Scientist and Engineer Experience AND Industry Alignment) OR
    (Industry = Renewable Energy & Climate Tech AND Versatility Across Multiple
    Industries AND Leadership Roles AND Long-Term Industry Experience AND Business
    Acumen) OR
(Educational Background FROM a Top 30 Globally Ranked Institution AND Professional
    Experience IN Product Management and Software Engineering in the tech industry AND
    Involvement in a Startup within a High-Growth Sector)) THEN label = YES.

For NO: IF ((Educational Background is MSc in Telecommunications Engineering From an
    institution ranked >200 OR Industry Focus is Wellness And Community Health With No
  

,rule,outcome,perplexity
0,IF Visual Effects Weakness AND Character Development Light AND Cinematic Imagery Limited AND Complex Themes Absent AND Narrative Structure Conventional AND Dialogue Basic THEN label = NO.,NO,1.0207
1,IF (bland premise AND overreliance on familiar tropes AND underdeveloped characters AND predictable dialogue AND lack of thematic depth AND weak pacing AND limited special effects AND poor execution of action sequences AND lack of emotional resonancy AND conventional storytelling techniques) THEN label = NO.,NO,1.0227
2,IF (Complex Narrative Structure AND Highly Visual and Violent Content AND Cinematographic Imagery AND Philosophical Depth AND Memorable Characters and Motivations AND Innovative Themes AND Symbolism and Meaningful Imagery AND Iconic Status) THEN label = YES.,YES,1.0261
3,IF (Simple Plot Structure AND Limited Focus AND Unremarkable Dialogue/Dialogue Delivery AND Visual/Aesthetic Weaknesses AND Avoids Heavy Footnote/Literary Praise AND Familiar Themes/Settings AND Emotional Impact Absence) THEN label = NO.,NO,1.0328
4,IF (Depth of Characterization AND Rich Historical Context AND Symbolism and Imagery AND Adult Themes for Young Audience AND Striking Visuals and Direction AND Strong Moral Themes AND Cinematic Influence AND Award-Winning Score) THEN label = YES.,YES,1.0331
5,IF (Complexity and Depth of Plot AND Intrigue and Suspense AND Character Development AND Themes and Symbolism AND Visual Imagery and Aesthetics AND Historical and Cultural Context AND Emotional Impact AND Riveting Central Performance) THEN label = YES.,YES,1.0344


RRM · VCBench public (n=4,500) · ROC-AUC 0.6665 · PR-AUC 0.1725


## The four side by side

The public column is computed from the shipped score files. The private column
is static: VCBench's private labels are deliberately not in this repo, so those
numbers cannot be recomputed here. The ensemble row is the rank-average of the
four methods that notebook 01 built.

Read the private column carefully.

GPTree loses most of its edge on held-out data (0.6161 → 0.5268, barely above
chance) while RRF and RRM barely move, so a good public number is not a promise.
And the ensemble, which wins on both public metrics and on private ROC-AUC, does
**not** win on private PR-AUC: RRF alone gets 0.2262 against the ensemble's
0.2129. Ensembling is a good default, not a clean sweep.

In [6]:
ensemble_public = pd.read_csv(PRECOMPUTED / "vcbench_public_pi_scores.csv").set_index("id")["label"]
wide = pd.DataFrame({
    name: pd.read_csv(PRECOMPUTED / f"vcbench_public_{key}_scores.csv").set_index("id")["score"]
    for name, key in [("PI", "pi"), ("RRF", "rrf"), ("GPTree", "gptree"), ("RRM", "rrm")]
}).reindex(ensemble_public.index)

# Percentile rank per method, then the mean of the four. pandas' default
# tie-handling (method="average") is the one the shipped numbers use.
public_metrics["Reasoning ensemble"] = score_ranking(
    ensemble_public, wide.rank(pct=True).mean(axis=1)
)

# Static reference: the same five series on the private split, from the
# maintainers' run. Not computable here, by design.
PRIVATE = {
    "PI": (0.6556, 0.1740),
    "RRF": (0.6681, 0.2262),
    "GPTree": (0.5268, 0.1053),
    "RRM": (0.6662, 0.1638),
    "Reasoning ensemble": (0.6850, 0.2129),
}

summary = pd.DataFrame(
    [
        {
            "Method": name,
            "Public ROC": metrics["ROC-AUC"],
            "Public PR": metrics["PR-AUC"],
            "Private ROC": PRIVATE[name][0],
            "Private PR": PRIVATE[name][1],
        }
        for name, metrics in public_metrics.items()
    ]
).set_index("Method")

# The public half is live, so it fails loudly if a load or an operator drifts.
REFERENCE_PUBLIC = {
    "PI": (0.6763, 0.1831),
    "RRF": (0.6604, 0.1689),
    "GPTree": (0.6161, 0.1320),
    "RRM": (0.6665, 0.1725),
    "Reasoning ensemble": (0.7188, 0.2072),
}
for name, (roc, pr) in REFERENCE_PUBLIC.items():
    row = summary.loc[name]
    assert abs(row["Public ROC"] - roc) < 1e-3, (name, "ROC", row["Public ROC"], roc)
    assert abs(row["Public PR"] - pr) < 1e-3, (name, "PR", row["Public PR"], pr)

summary.round(4)

,Public ROC,Public PR,Private ROC,Private PR
Method,,,,
PI,0.6763,0.1831,0.6556,0.1740
RRF,0.6604,0.1689,0.6681,0.2262
GPTree,0.6161,0.1320,0.5268,0.1053
RRM,0.6665,0.1725,0.6662,0.1638
Reasoning ensemble,0.7188,0.2072,0.6850,0.2129


## One live call

Everything above was read off disk. This cell talks to the model.

It builds the LLM exactly the way the runner scripts do, through
`src.llm.get_local_llm()`, and asks one question from the shipped Movie RRF
shortlist about a short plot summary. A full RRF pass over Movie makes about
30,000 of these (14 questions × 2,188 films), so this is a taste of the loop,
not a run.

The cell skips itself when nothing is listening on the Ollama port, so the
notebook still runs end to end without it. To run it for real:

```bash
ollama serve
ollama pull qwen2.5-coder:14b
```

To run a whole method instead of one question, use `scripts/run_pi.py`,
`scripts/run_rrf.py`, `scripts/run_gptree.py` or `scripts/run_rrm.py`. Those are
hours of local inference, not seconds.

In [ ]:
MODEL = "qwen2.5-coder:14b"
BASE_URL = os.environ.get("OPENAI_BASE_URL", "http://localhost:11434/v1")

# A stand-in plot summary in the shape of Movie's `summary` field. Written for
# this notebook, so the cell needs no dataset.
SYNOPSIS = (
    "A retired cartographer returns to the valley her village stood in before it "
    "was flooded, determined to map the place one last time from memory. Working "
    "from a single surviving photograph and the recollections of the neighbours "
    "she can still find, she redraws streets that no longer exist. As the map "
    "fills in, her own account of the night the dam gave way stops matching "
    "anybody else's."
)


def llm_reachable(base_url: str = BASE_URL, timeout: float = 1.0) -> bool:
    """Is anything listening where src.llm is going to send its requests?"""
    parsed = urlparse(base_url)
    try:
        with socket.create_connection((parsed.hostname, parsed.port or 80), timeout=timeout):
            return True
    except OSError:
        return False


if not llm_reachable():
    print(f"Nothing is listening at {BASE_URL}, so this cell is skipping itself.")
    print("Start Ollama (`ollama serve`), pull the model, and run the cell again.")
else:
    sys.path.insert(0, str(EXAMPLE_ROOT))
    from src.llm import get_local_llm

    question = json.loads(
        (MODELS / "movie" / "rrf" / "movie_rrf_questions_qwen14b.json").read_text()
    )[0]

    print(textwrap.fill(SYNOPSIS, 88), "\n")
    print("Q:", question["text"])

    try:
        answer = await get_local_llm().respond(
            query=f"PLOT SUMMARY\n{SYNOPSIS}\n\nQUESTION\n{question['text']}",
            llm_priority=[{"provider": "openai", "model": MODEL}],
            response_format=str,
            instructions=(
                "You are an expert film critic. Answer the question about the plot "
                "summary with YES or NO on the first line, then one sentence of "
                "justification."
            ),
            temperature=0.0,
        )
    except Exception as exc:  # a served port is not a served model
        print(f"\nA: the call failed ({type(exc).__name__}: {exc}).")
        print(f"Check that `ollama pull {MODEL}` has finished and that {BASE_URL} is Ollama.")
    else:
        print("A:", answer.response)
        print(f"\n({answer.total_tokens} tokens, {MODEL} via {BASE_URL})")

## Where next

- **[03 — Traditional baselines](03_traditional_baselines.ipynb)** builds the
  five-model sklearn suite these methods have to beat. On VCBench it is a
  higher bar than the four numbers above.
- **[04 — Ensembling](04_ensembling.ipynb)** puts the two families together,
  which is where the example's actual claim lives.

The artifacts you read here are the ones the `scripts/run_*.py` runners write.
Point them at your own data and you get your own policies, questions, tree and
rules.